# Final model v7 — full-release CICFlowMeter_out.csv (realism stress test)

The v6 architecture run directly on the **full official release** (`Resource/CICFlowMeter_out.csv`,
3,540,241 flows) instead of the curated 447,915-row `Data.csv` subset, with **Dst Port / Protocol
kept as native features** (every row carries its own — no consensus join, no `svc_ambiguous`).

**⚠️ READ BEFORE QUOTING:**
- **Benign is 97.5%** here vs 80.0% in the curated file. Levels are **NOT comparable** to
  v6 (0.6485) or the svc run (0.7460) — same traffic, different base rate. This notebook answers:
  *does the pipeline + service-identity gain survive at a realistic benign rate?*
- Gate precision is expected to drop by **base-rate arithmetic alone** (the load cell prints the
  fixed-rate expectation) — that is the point of the experiment, not a defect.
- Attack rows are identical to the curated file (all were kept there; verified in
  `results/cic_unsw_svc/`). Only the Benign volume changes.
- Runtime ≈ 30–60 min (gate cross-fit on ~2.8M rows × 5 folds); several GB RAM (float32).
  OOF is cached to `cache_oof_v7_full.npz`.
- `final_model_v6.ipynb` stays untouched as the reproducibility record of 0.6485.


In [1]:
import numpy as np, pandas as pd, time, gc, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import RobustScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

CFM = Path("Resource/CICFlowMeter_out.csv")
RS, EPS = 42, 1e-6
CAT = {"Benign": 0, "Analysis": 1, "Backdoor": 2, "DoS": 3, "Exploits": 4,
       "Fuzzers": 5, "Generic": 6, "Reconnaissance": 7, "Shellcode": 8, "Worms": 9}
NAMES = list(CAT.keys())

## 1. Data Collection — load the full release

Only `Dst Port`, `Protocol`, the 76 flow features and `Label` are read. `Flow ID`, the IPs and
`Timestamp` are excluded from the model on purpose: IPs are testbed endpoint identities
(memorisation, not behaviour), and using the timestamp belongs to a future *chronological-split*
experiment, not to the feature set.

In [2]:
t0 = time.time()
hdr = pd.read_csv(CFM, nrows=0).columns.tolist()
FEAT76 = hdr[7:-1]
assert len(FEAT76) == 76, len(FEAT76)
dtypes = {c: np.float32 for c in FEAT76}
dtypes["Dst Port"] = np.int32; dtypes["Protocol"] = np.int16
df = pd.read_csv(CFM, usecols=["Dst Port", "Protocol"] + FEAT76 + ["Label"], dtype=dtypes)
y_full = df["Label"].map(CAT)
assert y_full.notna().all(), "unmapped label present"
print("loaded %d x %d in %.0fs" % (df.shape[0], df.shape[1], time.time() - t0))

vc = y_full.value_counts().sort_index()
for k, v in vc.items():
    print("  %d %-16s %8d  %8.4f%%" % (k, NAMES[k], v, 100 * v / len(df)))
ben = float((y_full == 0).mean())
att = 1 - ben
print("\nBenign share %.2f%% (curated Data.csv: 80.0%%)" % (100 * ben))
prec_fixed = 0.996 * att / (0.996 * att + 0.0225 * ben)
print("gate precision IF per-row rates matched the curated run (recall .996, fpr .0225): %.3f (was 0.941)"
      % prec_fixed)
print("-> any precision above this line is the gate actually improving from 10x more Benign data")

loaded 3540241 x 79 in 20s
  0 Benign            3450658   97.4696%
  1 Analysis              385    0.0109%
  2 Backdoor              452    0.0128%
  3 DoS                  4467    0.1262%
  4 Exploits            30951    0.8743%
  5 Fuzzers             29613    0.8365%
  6 Generic              4632    0.1308%
  7 Reconnaissance      16735    0.4727%
  8 Shellcode            2102    0.0594%
  9 Worms                 246    0.0069%

Benign share 97.47% (curated Data.csv: 80.0%)
gate precision IF per-row rates matched the curated run (recall .996, fpr .0225): 0.535 (was 0.941)
-> any precision above this line is the gate actually improving from 10x more Benign data


## 2. Data Preparation — dedupe, prune, derived, service features

Dedupe key = **everything the model sees + label** (76 features + Dst Port + Protocol + Label):
two flows identical in the 76 features but on different ports are now genuinely different rows.
Prune and the 31 derived features are exactly `final_model_v6.ipynb`; the service columns are the
`results/cic_unsw_svc/` set minus `svc_ambiguous` (no consensus needed — ports are native here).

In [3]:
n0 = len(df)
df = df.drop_duplicates().reset_index(drop=True)
y = df["Label"].map(CAT).astype(int).values
print("dedupe: %d -> %d (%.2f%% duplicates; curated file was 31.6%%)"
      % (n0, len(df), 100 * (1 - len(df) / n0)))
vc = pd.Series(y).value_counts().sort_index()
for k, v in vc.items():
    print("  %d %-16s %8d  %8.4f%%" % (k, NAMES[k], v, 100 * v / len(y)))

Xraw = df[FEAT76]
nun = Xraw.nunique(); zero = nun[nun <= 1].index.tolist()
FULL = Xraw.drop(columns=zero)
corr = FULL.corr(); upper = corr.abs().where(np.triu(np.ones(corr.shape, bool), k=1))
red = [c for c in upper.columns if (upper[c] >= 0.95).any()]
BASE = FULL.drop(columns=red)
print("prune: 76 -> %d (zero-var %d, |r|>=0.95 %d; curated file gave 44)"
      % (BASE.shape[1], len(zero), len(red)))

dedupe: 3540241 -> 2606803 (26.37% duplicates; curated file was 31.6%)
  0 Benign            2525314   96.8740%
  1 Analysis              385    0.0148%
  2 Backdoor              452    0.0173%
  3 DoS                  4418    0.1695%
  4 Exploits            30925    1.1863%
  5 Fuzzers             26433    1.0140%
  6 Generic              4496    0.1725%
  7 Reconnaissance      12035    0.4617%
  8 Shellcode            2102    0.0806%
  9 Worms                 243    0.0093%
prune: 76 -> 43 (zero-var 9, |r|>=0.95 24; curated file gave 44)


In [4]:
def safe(a, b):
    return (a / (b + EPS)).replace([np.inf, -np.inf], 0.0).fillna(0.0)

# identical construction to final_model_v6.ipynb / feature_engineering.ipynb
F = FULL
g = lambda c: F[c] if c in F.columns else pd.Series(0.0, index=F.index)
fpk, bpk = g("Total Fwd Packet"), g("Total Bwd packets")
fby, bby = g("Total Length of Fwd Packet"), g("Total Length of Bwd Packet")
tot_pk, tot_by = fpk + bpk, fby + bby
d = {}
d["r_pkt_fwd_bwd"] = safe(fpk, bpk); d["r_byte_fwd_bwd"] = safe(fby, bby)
d["asym_pkt"] = safe(fpk - bpk, tot_pk); d["asym_byte"] = safe(fby - bby, tot_by)
d["bpp_fwd"] = safe(fby, fpk); d["bpp_bwd"] = safe(bby, bpk); d["bpp_flow"] = safe(tot_by, tot_pk)
d["r_bpp"] = safe(d["bpp_fwd"], d["bpp_bwd"])
d["cv_flow_iat"] = safe(g("Flow IAT Std"), g("Flow IAT Mean"))
d["cv_fwd_iat"] = safe(g("Fwd IAT Std"), g("Fwd IAT Mean"))
d["cv_bwd_iat"] = safe(g("Bwd IAT Std"), g("Bwd IAT Mean"))
d["iat_span_rel"] = safe(g("Flow IAT Max") - g("Flow IAT Min"), g("Flow IAT Mean"))
d["cv_fwd_len"] = safe(g("Fwd Packet Length Std"), g("Fwd Packet Length Mean"))
d["cv_bwd_len"] = safe(g("Bwd Packet Length Std"), g("Bwd Packet Length Mean"))
d["cv_pkt_len"] = safe(g("Packet Length Std"), g("Packet Length Mean"))
d["fwd_len_span_rel"] = safe(g("Fwd Packet Length Max") - g("Fwd Packet Length Min"), g("Fwd Packet Length Mean"))
d["bwd_len_span_rel"] = safe(g("Bwd Packet Length Max") - g("Bwd Packet Length Min"), g("Bwd Packet Length Mean"))
for flag in ("FIN", "SYN", "RST", "PSH", "ACK"):
    d["frac_" + flag] = safe(g(flag + " Flag Count"), tot_pk)
d["r_syn_fin"] = safe(g("SYN Flag Count"), g("FIN Flag Count"))
d["hdr_per_pkt_fwd"] = safe(g("Fwd Header Length"), fpk)
d["hdr_per_pkt_bwd"] = safe(g("Bwd Header Length"), bpk)
d["payload_share_fwd"] = safe(fby - g("Fwd Header Length"), fby)
dur = g("Flow Duration")
d["active_share"] = safe(g("Active Mean"), dur); d["idle_share"] = safe(g("Idle Mean"), dur)
d["r_active_idle"] = safe(g("Active Mean"), g("Idle Mean")); d["dur_per_pkt"] = safe(dur, tot_pk)
d["r_init_win"] = safe(g("FWD Init Win Bytes"), g("Bwd Init Win Bytes"))
DER = pd.DataFrame(d, index=F.index).replace([np.inf, -np.inf], 0.0).fillna(0.0).astype(np.float32)
dz = DER.nunique(); DER = DER.drop(columns=dz[dz <= 1].index.tolist())
del d, fpk, bpk, fby, bby, tot_pk, tot_by, dur; gc.collect()
print("derived: %d columns" % DER.shape[1])

# --- service columns: native per-row port/protocol (no consensus, no svc_ambiguous) ---
port = df["Dst Port"].values
proto = df["Protocol"].values
SVC_PORTS = [(20, "svc_ftpdata"), (21, "svc_ftp"), (22, "svc_ssh"), (23, "svc_telnet"),
             (25, "svc_smtp"), (53, "svc_dns"), (80, "svc_http"), (110, "svc_pop3"),
             (111, "svc_rpc"), (139, "svc_netbios"), (143, "svc_imap"), (179, "svc_bgp"),
             (443, "svc_https"), (445, "svc_smb"), (514, "svc_syslog"), (520, "svc_rip"),
             (1723, "svc_pptp"), (5190, "svc_aim"), (6881, "svc_bt"), (8080, "svc_httpalt")]
KNOWN = np.array([p for p, _ in SVC_PORTS])
s = {nm: (port == p).astype(np.float32) for p, nm in SVC_PORTS}
s["svc_other_low"] = ((port < 1024) & ~np.isin(port, KNOWN)).astype(np.float32)
s["svc_registered"] = ((port >= 1024) & (port < 49152) & ~np.isin(port, KNOWN)).astype(np.float32)
s["svc_ephemeral"] = (port >= 49152).astype(np.float32)
s["proto_tcp"] = (proto == 6).astype(np.float32)
s["proto_udp"] = (proto == 17).astype(np.float32)
s["dst_port_num"] = port.astype(np.float32)
SVC = pd.DataFrame(s, index=df.index)
sz = SVC.nunique(); SVC = SVC.drop(columns=sz[sz <= 1].index.tolist())
SVC_NAMES = set(SVC.columns); DER_NAMES = set(DER.columns)
X = pd.concat([BASE.astype(np.float32), DER, SVC], axis=1)
COLS = list(X.columns)
print("v7 feature space: %d = %d BASE + %d derived + %d svc"
      % (X.shape[1], BASE.shape[1], DER.shape[1], SVC.shape[1]))
del Xraw, FULL, BASE, DER, SVC, corr, upper, s, df; gc.collect()
print("X memory: %.1f GB" % (X.memory_usage().sum() / 1e9))

derived: 31 columns
v7 feature space: 99 = 43 BASE + 31 derived + 25 svc
X memory: 1.0 GB


### 2.7 Stratified split (80/20), family targets, COMB/SELV halves — v6 protocol, v6 seeds

In [5]:
idx_all = np.arange(len(y))
i_trf, i_test = train_test_split(idx_all, test_size=0.2, random_state=RS, stratify=y)
X_trf = X.iloc[i_trf].reset_index(drop=True)
X_test = X.iloc[i_test].reset_index(drop=True)
y_arr = y[i_trf]; y_te = y[i_test]
del X; gc.collect()

attack_labels = sorted(l for l in np.unique(y_arr) if l != 0)
fam_to_idx = {f: i for i, f in enumerate(attack_labels)}
idx_to_fam = {i: f for f, i in fam_to_idx.items()}
NC = len(attack_labels)
att_trf = y_arr != 0
fam_trf = np.full(len(y_arr), -1)
fam_trf[att_trf] = pd.Series(y_arr[att_trf]).map(fam_to_idx).values
att_te = y_te != 0
fam_te = pd.Series(y_te[att_te]).map(fam_to_idx).values

comb_idx, selv_idx = train_test_split(np.arange(len(y_arr)), test_size=0.5,
                                      random_state=RS + 1, stratify=y_arr)
ref_idx = comb_idx[att_trf[comb_idx]]
y_ref = fam_trf[ref_idx]
print("train+val %d (%d attack) | test %d (%d attack)"
      % (len(y_arr), att_trf.sum(), len(y_te), att_te.sum()))
print("COMB attack reference %d | SELV %d" % (len(ref_idx), len(selv_idx)))

train+val 2085442 (65191 attack) | test 521361 (16298 attack)
COMB attack reference 32596 | SELV 1042721


### 2.8 Cover–Hart bracket on attack rows (metric-relative, as always)

In [6]:
sc = RobustScaler().fit(X_trf[att_trf])
Ztr = sc.transform(X_trf[att_trf]).astype(np.float32)
Zte = sc.transform(X_test[att_te]).astype(np.float32)
t0 = time.time()
nn1 = NearestNeighbors(n_neighbors=1, algorithm="brute", n_jobs=-1).fit(Ztr)
_, ii1 = nn1.kneighbors(Zte)
e1 = float((fam_trf[att_trf][ii1[:, 0]] != fam_te).mean())
ch = (NC - 1) / NC * (1 - np.sqrt(max(0.0, 1 - NC / (NC - 1) * e1)))
print("e_1NN %.4f -> e* in [%.4f, %.4f]  (%.0fs)" % (e1, ch, e1, time.time() - t0))
print("curated-file references: DERIVED 0.3632 | DERIVED_SVC 0.2098")
print("(comparable in spirit only -- the attack rows are the same, but dedupe differs: rows")
print(" identical in 76 features now survive when their ports differ)")
del Ztr, Zte, nn1; gc.collect()

e_1NN 0.2084 -> e* in [0.1112, 0.2084]  (1s)
curated-file references: DERIVED 0.3632 | DERIVED_SVC 0.2098
(comparable in spirit only -- the attack rows are the same, but dedupe differs: rows
 identical in 76 features now survive when their ports differ)


23

## 3–4. Model Selection & Training — v6 pool, 5-fold cross-fit

Gate = binary XGB (`balanced`); Stage B = 7-member α-diverse pool. Verbatim v6 machinery.
The gate folds fit on ~2.3M rows each — this is the slow cell (expect ~20–40 min total).
OOF is cached; re-running the notebook skips it.

In [7]:
def make_xgb(**kw):
    p = dict(n_estimators=400, max_depth=8, learning_rate=0.1, subsample=0.9,
             colsample_bytree=0.9, tree_method="hist", eval_metric="mlogloss",
             n_jobs=-1, random_state=RS)
    p.update(kw); return XGBClassifier(**p)

def wc_alpha(n, a):
    w = (n.sum() / n) ** a; return w / w.mean()

def sample_w(ya, wc):
    w = wc[ya].astype(float); return w * len(w) / w.sum()

SPECS = ([(f"XGB a={a}", ("alpha", a)) for a in (0.0, 0.25, 0.5, 1.0)]
         + [("XGB smote2000", ("smote", 2000)),
            ("RF a=0.25", ("rf", 0.25)), ("ET a=0.25", ("et", 0.25))])
MODELS = [s_[0] for s_ in SPECS]; M = len(MODELS)
I25 = MODELS.index("XGB a=0.25")

def fit_member(kind, param, Xa, ya):
    n = np.bincount(np.asarray(ya), minlength=NC)
    if kind == "smote":
        strat = {c: param for c in range(NC) if n[c] < param}
        Xs, ys = SMOTE(random_state=RS, sampling_strategy=strat).fit_resample(Xa, ya)
        m = make_xgb(); m.fit(Xs, ys); return m
    if kind == "alpha":  m = make_xgb()
    elif kind == "rf":   m = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RS)
    elif kind == "et":   m = ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=RS)
    m.fit(Xa, ya, sample_weight=sample_w(np.asarray(ya), wc_alpha(n, param)))
    return m

def fit_gate(Xa, ya10):
    yb = (ya10 != 0).astype(int)
    g_ = make_xgb(); g_.fit(Xa, yb, sample_weight=compute_sample_weight("balanced", yb))
    return g_

print("pool:", M, "members")

pool: 7 members


In [8]:
CACHE = Path("cache_oof_v7_full.npz")
if CACHE.exists():
    z = np.load(CACHE)
    OOF = [z[f"m{m}"] for m in range(M)]; oof_gate = z["gate"]
    print("loaded cached OOF -", CACHE)
else:
    OOF = [np.zeros((len(y_arr), NC), np.float32) for _ in range(M)]
    oof_gate = np.zeros(len(y_arr), np.int8)
    t0 = time.time()
    for f_, (tr, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=RS).split(X_trf, y_arr)):
        tf = time.time()
        Xtr_f, ytr_f = X_trf.iloc[tr], y_arr[tr]
        gate_m = fit_gate(Xtr_f, ytr_f)
        oof_gate[va] = gate_m.predict(X_trf.iloc[va])
        del gate_m; gc.collect()
        print("  fold %d gate done (%.0fs)" % (f_ + 1, time.time() - tf))
        m_att = ytr_f != 0
        Xa = Xtr_f[m_att].reset_index(drop=True)
        ya = pd.Series(fam_trf[tr][m_att])
        for mi, (nm, (kind, param)) in enumerate(SPECS):
            mdl = fit_member(kind, param, Xa, ya)
            OOF[mi][va] = mdl.predict_proba(X_trf.iloc[va]).astype(np.float32)
            del mdl; gc.collect()
        print("  fold %d/5 done (%.0fs, elapsed %.0fm)"
              % (f_ + 1, time.time() - tf, (time.time() - t0) / 60))
    print("cross-fit complete in %.0fm" % ((time.time() - t0) / 60))
    np.savez_compressed(CACHE, gate=oof_gate, **{f"m{m}": OOF[m] for m in range(M)})

yb_tr = (y_arr != 0).astype(int)
gr_, gp_ = recall_score(yb_tr, oof_gate), precision_score(yb_tr, oof_gate)
print("OOF gate: attack recall %.4f / precision %.4f" % (gr_, gp_))
print("  fixed-rate expectation was precision ~%.3f; curated-file gate was 0.9414" % prec_fixed)

  fold 1 gate done (30s)
  fold 1/5 done (139s, elapsed 2m)
  fold 2 gate done (29s)
  fold 2/5 done (144s, elapsed 5m)
  fold 3 gate done (31s)
  fold 3/5 done (135s, elapsed 7m)
  fold 4 gate done (29s)
  fold 4/5 done (140s, elapsed 9m)
  fold 5 gate done (30s)
  fold 5/5 done (136s, elapsed 12m)
cross-fit complete in 12m
OOF gate: attack recall 0.9880 / precision 0.6538
  fixed-rate expectation was precision ~0.535; curated-file gate was 0.9414


## 5–6. Combiner (per-class W + decision-space dynamic), SELV selection

In [9]:
CORRECT = np.stack([(OOF[m][ref_idx].argmax(1) == y_ref) for m in range(M)])
WC_REF = wc_alpha(np.bincount(y_ref, minlength=NC), 0.25)
prof = lambda ps: np.hstack(ps).astype(np.float32)
NN_D = NearestNeighbors(n_neighbors=100, algorithm="brute", n_jobs=-1).fit(
    prof([OOF[m][ref_idx] for m in range(M)]))

def comp_ola(idx, k):
    nb = idx[:, :k]; wn = WC_REF[y_ref[nb]]; den = wn.sum(1)
    W = np.empty((nb.shape[0], M))
    for m in range(M):
        W[:, m] = (CORRECT[m][nb] * wn).sum(1) / den
    return W

def comp_lca(probas, idx, k):
    nb = idx[:, :k]; yn = y_ref[nb]; W = np.empty((nb.shape[0], M))
    for m in range(M):
        pq = probas[m].argmax(1); mask = yn == pq[:, None]; cnt = mask.sum(1)
        W[:, m] = np.where(cnt > 0, (CORRECT[m][nb] & mask).sum(1) / np.maximum(cnt, 1),
                           CORRECT[m][nb].mean(1))
    return W

def fuse_dyn(probas, Wd, p=1.0, Wclass=None):
    Wp = np.power(np.clip(Wd, 1e-6, None), p); Wp = Wp / Wp.sum(1, keepdims=True)
    S = np.zeros(probas[0].shape, np.float64)
    for m in range(M):
        coef = Wp[:, m:m + 1] if Wclass is None else Wp[:, m:m + 1] * Wclass[m][None, :]
        S += coef * probas[m]
    return S.argmax(1)

def fuse_static(probas, Wclass):
    S = np.zeros(probas[0].shape, np.float64)
    for m in range(M):
        S += probas[m] * Wclass[m][None, :]
    return S.argmax(1)

GRID_W = (0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0)
def fit_W(probas, ytrue, passes=3):
    W = np.ones((M, NC)) / M
    best = f1_score(ytrue, fuse_static(probas, W), average="macro", zero_division=0)
    for _ in range(passes):
        for c in range(NC):
            for m_ in range(M):
                cur = W[m_, c]
                for gg in GRID_W:
                    W[m_, c] = gg
                    if W[:, c].sum() == 0: continue
                    s_ = f1_score(ytrue, fuse_static(probas, W), average="macro", zero_division=0)
                    if s_ > best + 1e-7: best, cur = s_, gg
                W[m_, c] = cur
    return W, best

t0 = time.time()
W_class, s_comb = fit_W([OOF[m][ref_idx] for m in range(M)], y_ref)
print("W fitted in %.0fs -> COMB Stage-B mF1 %.4f" % (time.time() - t0, s_comb))

W fitted in 14s -> COMB Stage-B mF1 0.7626


In [10]:
def score10_g(fam_g, ye, gmask, tag):
    yp = np.zeros(len(ye), int)
    if len(fam_g):
        yp[gmask] = np.vectorize(idx_to_fam.get, otypes=[int])(fam_g)
    return dict(variant=tag,
                e2e_mF1=round(f1_score(ye, yp, average="macro", zero_division=0), 4),
                e2e_mP=round(precision_score(ye, yp, average="macro", zero_division=0), 4),
                e2e_mR=round(recall_score(ye, yp, average="macro", zero_division=0), 4),
                e2e_acc=round(float((yp == ye).mean()), 4))

def run_all(P_g, idx, ye, gmask):
    out = [score10_g(fuse_static(P_g, W_class), ye, gmask, "STATIC: per-class W"),
           score10_g(fuse_static(P_g, np.ones((M, NC)) / M), ye, gmask, "STATIC: equal average")]
    for m in range(M):
        out.append(score10_g(P_g[m].argmax(1), ye, gmask, "single: " + MODELS[m]))
    for k in (30, 100):
        for p in (1, 3, 8):
            out.append(score10_g(fuse_dyn(P_g, comp_ola(idx, k), p=p), ye, gmask,
                                 f"DYN OLA-cw k={k} p={p}"))
    out.append(score10_g(fuse_dyn(P_g, comp_lca(P_g, idx, 30)), ye, gmask, "DYN LCA k=30"))
    for k in (30, 100):
        for p in (1, 3):
            out.append(score10_g(fuse_dyn(P_g, comp_ola(idx, k), p=p, Wclass=W_class), ye, gmask,
                                 f"HYBRID W x OLA-cw k={k} p={p}"))
    out.append(score10_g(fuse_dyn(P_g, comp_lca(P_g, idx, 30), p=1, Wclass=W_class), ye, gmask,
                         "HYBRID W x LCA k=30 p=1"))
    return pd.DataFrame(out)

gmask_selv = oof_gate[selv_idx] == 1
gpos_selv = selv_idx[gmask_selv]
P_selv = [OOF[m][gpos_selv] for m in range(M)]
t0 = time.time()
_, i_selv = NN_D.kneighbors(prof(P_selv))
print("SELV kNN: %d gate-positive rows in %.0fs" % (len(gpos_selv), time.time() - t0))

selv_tab = run_all(P_selv, i_selv, y_arr[selv_idx], gmask_selv).sort_values(
    "e2e_mF1", ascending=False).reset_index(drop=True)
print(selv_tab.to_string(index=False))
BESTV = selv_tab.iloc[0]["variant"]
print("\nSELV winner: %s (%.4f)" % (BESTV, selv_tab.iloc[0]["e2e_mF1"]))
del P_selv, i_selv; gc.collect()

SELV kNN: 49379 gate-positive rows in 1s
                    variant  e2e_mF1  e2e_mP  e2e_mR  e2e_acc
        DYN OLA-cw k=30 p=1   0.6449  0.6407  0.7165   0.9793
        DYN OLA-cw k=30 p=3   0.6448  0.6372  0.7215   0.9794
       DYN OLA-cw k=100 p=3   0.6448  0.6402  0.7186   0.9793
HYBRID W x OLA-cw k=100 p=3   0.6445  0.6259  0.7382   0.9794
       DYN OLA-cw k=100 p=8   0.6445  0.6360  0.7246   0.9794
 HYBRID W x OLA-cw k=30 p=3   0.6442  0.6246  0.7389   0.9794
               DYN LCA k=30   0.6440  0.6462  0.7098   0.9793
 HYBRID W x OLA-cw k=30 p=1   0.6439  0.6248  0.7375   0.9793
      STATIC: equal average   0.6433  0.6456  0.7080   0.9793
        DYN OLA-cw k=30 p=8   0.6430  0.6303  0.7255   0.9794
       DYN OLA-cw k=100 p=1   0.6429  0.6416  0.7119   0.9793
HYBRID W x OLA-cw k=100 p=1   0.6418  0.6232  0.7349   0.9793
    HYBRID W x LCA k=30 p=1   0.6415  0.6248  0.7323   0.9793
        STATIC: per-class W   0.6410  0.6235  0.7331   0.9793
          single: RF a=0.25  

0

## 7. Refit on train+val, test ONCE

In [11]:
t0 = time.time()
gate_f = fit_gate(X_trf, y_arr)
Xa_f = X_trf[att_trf].reset_index(drop=True)
ya_f = pd.Series(fam_trf[att_trf])
FINAL = [fit_member(k, p, Xa_f, ya_f) for _, (k, p) in SPECS]
print("pool + gate refit in %.0fm" % ((time.time() - t0) / 60))

gmask_test = gate_f.predict(X_test) == 1
gpos_test = np.where(gmask_test)[0]
P_test = [m.predict_proba(X_test.iloc[gpos_test]).astype(np.float32) for m in FINAL]
_, i_test_nn = NN_D.kneighbors(prof(P_test))

test_tab = run_all(P_test, i_test_nn, y_te, gmask_test).sort_values(
    "e2e_mF1", ascending=False).reset_index(drop=True)
print("\n--- TEST (selection already made on SELV) ---")
print(test_tab.to_string(index=False))
sel = test_tab[test_tab.variant == BESTV].iloc[0]
rank = 1 + list(test_tab.variant).index(BESTV)
print("\nSELV-selected '%s' ranks %d of %d on test" % (BESTV, rank, len(test_tab)))
print("v7-full: macro-F1 %.4f | accuracy %.4f" % (sel["e2e_mF1"], sel["e2e_acc"]))
print("NOT level-comparable to v6 0.6485 or svc-run 0.7460 (Benign 97.5%% vs 80%%)")

pool + gate refit in 2m

--- TEST (selection already made on SELV) ---
                    variant  e2e_mF1  e2e_mP  e2e_mR  e2e_acc
        DYN OLA-cw k=30 p=3   0.6619  0.6550  0.7410   0.9799
 HYBRID W x OLA-cw k=30 p=3   0.6608  0.6434  0.7549   0.9799
               DYN LCA k=30   0.6607  0.6618  0.7272   0.9799
HYBRID W x OLA-cw k=100 p=1   0.6606  0.6445  0.7539   0.9799
 HYBRID W x OLA-cw k=30 p=1   0.6604  0.6442  0.7540   0.9799
        DYN OLA-cw k=30 p=8   0.6603  0.6487  0.7455   0.9799
         single: XGB a=0.25   0.6603  0.6559  0.7300   0.9798
        STATIC: per-class W   0.6602  0.6440  0.7534   0.9799
    HYBRID W x LCA k=30 p=1   0.6602  0.6447  0.7506   0.9799
        DYN OLA-cw k=30 p=1   0.6601  0.6568  0.7329   0.9799
      STATIC: equal average   0.6594  0.6565  0.7275   0.9799
       DYN OLA-cw k=100 p=1   0.6591  0.6549  0.7299   0.9799
       DYN OLA-cw k=100 p=3   0.6590  0.6526  0.7363   0.9799
       DYN OLA-cw k=100 p=8   0.6590  0.6500  0.7426   0.9799

In [12]:
def build(t, P_g, idx):
    if t == "STATIC: per-class W":   return fuse_static(P_g, W_class)
    if t == "STATIC: equal average": return fuse_static(P_g, np.ones((M, NC)) / M)
    if t.startswith("single: "):     return P_g[MODELS.index(t[8:])].argmax(1)
    if t == "DYN LCA k=30":          return fuse_dyn(P_g, comp_lca(P_g, idx, 30))
    if t == "HYBRID W x LCA k=30 p=1":
        return fuse_dyn(P_g, comp_lca(P_g, idx, 30), p=1, Wclass=W_class)
    k = 100 if "k=100" in t else 30
    p = 8 if "p=8" in t else (3 if "p=3" in t else 1)
    return fuse_dyn(P_g, comp_ola(idx, k), p=p,
                    Wclass=(W_class if t.startswith("HYBRID") else None))

fam_pred = build(BESTV, P_test, i_test_nn)
yp = np.zeros(len(y_te), int)
yp[gmask_test] = np.vectorize(idx_to_fam.get, otypes=[int])(fam_pred)
print(classification_report(y_te, yp, labels=list(range(10)),
                            target_names=NAMES, digits=3, zero_division=0))
print(pd.DataFrame(confusion_matrix(y_te, yp, labels=list(range(10))),
                   index=NAMES, columns=NAMES).to_string())

gr = recall_score((y_te != 0).astype(int), gmask_test.astype(int))
gp = precision_score((y_te != 0).astype(int), gmask_test.astype(int))
print("\ngate on test: attack recall %.4f / precision %.4f" % (gr, gp))
print("  fixed-rate expectation %.3f | curated-file svc gate 0.9411" % prec_fixed)

                precision    recall  f1-score   support

        Benign      1.000     0.983     0.991    505063
      Analysis      0.103     0.481     0.169        77
      Backdoor      0.823     0.567     0.671        90
           DoS      0.687     0.484     0.568       884
      Exploits      0.704     0.907     0.793      6185
       Fuzzers      0.438     0.943     0.598      5287
       Generic      0.880     0.888     0.884       899
Reconnaissance      0.831     0.802     0.816      2407
     Shellcode      0.389     0.764     0.515       420
         Worms      0.714     0.510     0.595        49

      accuracy                          0.980    521361
     macro avg      0.657     0.733     0.660    521361
  weighted avg      0.988     0.980     0.983    521361

                Benign  Analysis  Backdoor  DoS  Exploits  Fuzzers  Generic  Reconnaissance  Shellcode  Worms
Benign          496693       280         3   87      1118     6327       38              90        426 

### 7.1 Marginal bootstrap CI (2,000 stratified resamples) and per-class comparison

Per-class F1 against the curated-file svc run — **direction only**, never levels: the per-family
attack rows are the same traffic, but Benign volume (and hence precision pressure) is 12× higher.

In [13]:
def macro_f1_from_counts(cm):
    tp = np.diag(cm).astype(float)
    fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    return (2 * tp / np.maximum(2 * tp + fp + fn, 1e-12)).mean()

code_v = y_te * 10 + yp
rng = np.random.RandomState(RS)
cls_idx = [np.where(y_te == c)[0] for c in range(10)]
f1s = np.empty(2000)
for it in range(2000):
    bidx = np.concatenate([ci[rng.randint(0, len(ci), len(ci))] for ci in cls_idx])
    f1s[it] = macro_f1_from_counts(np.bincount(code_v[bidx], minlength=100).reshape(10, 10))
lo, hi = np.percentile(f1s, [2.5, 97.5])
print("v7-full macro-F1 %.4f [%.4f, %.4f]" % (sel["e2e_mF1"], lo, hi))

SVC_RUN = {"Benign": 0.9880, "Analysis": 0.4519, "Backdoor": 0.7034, "DoS": 0.5601,
           "Exploits": 0.8499, "Fuzzers": 0.8989, "Generic": 0.8842,
           "Reconnaissance": 0.8327, "Shellcode": 0.6579, "Worms": 0.6329}
f1c = f1_score(y_te, yp, average=None, labels=range(10), zero_division=0)
print("\nper-class F1: v7-full vs curated svc run (direction only, NOT levels):")
for i, n in enumerate(NAMES):
    print("  %-16s %.4f  (svc run %.4f, %+.4f)" % (n, f1c[i], SVC_RUN[n], f1c[i] - SVC_RUN[n]))

imp = pd.Series(FINAL[I25].feature_importances_, index=COLS).sort_values(ascending=False)
print("\ntop 20 Stage-B features (XGB a=0.25):")
for i, (kk, vv) in enumerate(imp.head(20).items()):
    mark = "(svc)" if kk in SVC_NAMES else ("(der)" if kk in DER_NAMES else "")
    print("  %2d. %-24s %.4f %s" % (i + 1, kk, vv, mark))
svc_share = imp[[c for c in imp.index if c in SVC_NAMES]].sum() / imp.sum()
der_share = imp[[c for c in imp.index if c in DER_NAMES]].sum() / imp.sum()
print("svc = %.1f%% of importance (curated run: 70.4%%) | derived = %.1f%%"
      % (100 * svc_share, 100 * der_share))

v7-full macro-F1 0.6601 [0.6430, 0.6765]

per-class F1: v7-full vs curated svc run (direction only, NOT levels):
  Benign           0.9915  (svc run 0.9880, +0.0035)
  Analysis         0.1693  (svc run 0.4519, -0.2826)
  Backdoor         0.6711  (svc run 0.7034, -0.0323)
  DoS              0.5680  (svc run 0.5601, +0.0079)
  Exploits         0.7927  (svc run 0.8499, -0.0572)
  Fuzzers          0.5977  (svc run 0.8989, -0.3012)
  Generic          0.8837  (svc run 0.8842, -0.0005)
  Reconnaissance   0.8165  (svc run 0.8327, -0.0162)
  Shellcode        0.5152  (svc run 0.6579, -0.1427)
  Worms            0.5952  (svc run 0.6329, -0.0377)

top 20 Stage-B features (XGB a=0.25):
   1. svc_rpc                  0.4183 (svc)
   2. svc_bgp                  0.0473 (svc)
   3. frac_PSH                 0.0300 (der)
   4. svc_pptp                 0.0240 (svc)
   5. svc_http                 0.0239 (svc)
   6. svc_other_low            0.0235 (svc)
   7. svc_imap                 0.0217 (svc)
   8. Bwd 